# 1. Mã hóa biến cơ bản 

## 2.1 Mục tiêu của mã hóa biến
Trong học máy, phần lớn mô hình chỉ làm việc trực tiếp với dữ liệu số. Vì vậy, các biến phân loại (categorical variables) cần được chuyển đổi sang dạng số để mô hình có thể học được.

Hai kỹ thuật cơ bản và phổ biến nhất là:
- **One-hot Encoding**
- **Ordinal Encoding**



## 1.2 Phân loại biến phân loại trước khi mã hóa
Trước khi chọn kỹ thuật mã hóa, cần xác định kiểu biến:

- **Nominal (không có thứ tự)**: các nhóm chỉ khác nhau về nhãn, không có quan hệ lớn hơn/nhỏ hơn.
  - Ví dụ: màu sắc (`red`, `blue`, `green`), thành phố, loại trình duyệt.
- **Ordinal (có thứ tự)**: các nhóm có thứ tự rõ ràng.
  - Ví dụ: mức hài lòng (`low < medium < high`), hạng vé (`economy < business < first`).


## 1.3 One-hot Encoding
### Định nghĩa
One-hot Encoding biến mỗi giá trị phân loại thành một cột nhị phân riêng (0/1).

Ví dụ cột `color = [red, blue, green]` sẽ thành:
- `color_red`, `color_blue`, `color_green`
- Mỗi dòng chỉ có một cột bằng 1, các cột còn lại bằng 0.

### Ưu điểm
- Không áp đặt thứ tự giả giữa các nhóm.
- Dễ hiểu, trực quan, giải thích kết quả tốt.
- Thường hoạt động ổn với nhiều mô hình tuyến tính hoặc khoảng cách khi dữ liệu không quá lớn.

### Hạn chế
- Tăng mạnh số chiều dữ liệu khi biến có nhiều giá trị khác nhau (high cardinality).
- Dễ tạo ma trận thưa (sparse), tốn RAM/thời gian huấn luyện.
- Với tập test có nhãn mới (unseen category) cần cấu hình xử lý phù hợp (ví dụ `handle_unknown='ignore'`).
- Bị confused với giá trị 0.

### Dùng khi nào?
- Biến là **nominal**, không có thứ tự tự nhiên.
- Số lượng giá trị phân loại thấp đến trung bình.
- Cần mô hình dễ diễn giải.

### Phù hợp với bộ dữ liệu nào?
- Dữ liệu khảo sát, dữ liệu khách hàng, dữ liệu giao dịch có thuộc tính dạng loại như giới tính, khu vực, loại sản phẩm.
- Tốt nhất khi mỗi cột phân loại không có quá nhiều mức (ví dụ vài chục mức trở xuống).



## 1.4 Ordinal Encoding
### Định nghĩa
Ordinal Encoding ánh xạ mỗi mức phân loại thành một số nguyên theo thứ tự đã biết.

Ví dụ:
- `size: small < medium < large`
- Mã hóa thành: `small=0`, `medium=1`, `large=2`.

### Ưu điểm
- Rất gọn, chỉ tạo 1 cột số cho mỗi biến phân loại.
- Giữ được thông tin thứ tự của dữ liệu.
- Hiệu quả về bộ nhớ và tốc độ hơn One-hot khi số mức lớn.

### Hạn chế
- Nếu dùng cho biến **nominal** sẽ tạo quan hệ thứ tự sai, làm mô hình hiểu lệch bản chất dữ liệu.
- Khoảng cách số (ví dụ 0, 1, 2) có thể bị mô hình diễn giải như khoảng cách thật, trong khi thực tế có thể không tuyến tính.
- Cần định nghĩa thứ tự chính xác theo nghiệp vụ.

### Dùng khi nào?
- Biến có **thứ tự tự nhiên** rõ ràng.
- Muốn giảm số chiều dữ liệu.
- Phù hợp khi mô hình hoặc bài toán cần giữ quan hệ tăng/giảm giữa các mức.

### Phù hợp với bộ dữ liệu nào?
- Dữ liệu đánh giá mức độ, chấm điểm, xếp hạng, cấp độ rủi ro, mức ưu tiên.
- Các tập dữ liệu có cột phân loại dạng cấp bậc rõ ràng.



## 1.5 So sánh nhanh One-hot vs Ordinal
| Tiêu chí | One-hot Encoding | Ordinal Encoding |
|---|---|---|
| Loại biến phù hợp | Nominal | Ordinal |
| Số cột tạo ra | Nhiều cột (theo số mức) | 1 cột |
| Giả định thứ tự | Không | Có |
| Rủi ro chính | Bùng nổ số chiều | Áp thứ tự sai nếu dùng nhầm |
| Khả năng diễn giải | Cao | Trung bình (phụ thuộc mapping) |


# 2. Mã hóa nâng cao 

Các kỹ thuật mã hóa nâng cao thường được dùng khi:
- Cột phân loại có nhiều giá trị khác nhau (high-cardinality).
- One-hot gây bùng nổ số chiều.
- Cần trích xuất thêm thông tin thống kê từ dữ liệu.

## 2.1 Target Encoding (Mean Encoding)

### Định nghĩa
Target Encoding thay mỗi giá trị phân loại bằng thống kê của biến mục tiêu (target), phổ biến nhất là **trung bình target theo từng nhóm**.

Ví dụ: nếu cột `district` có nhiều quận/huyện, mỗi quận sẽ được thay bằng trung bình giá nhà hoặc xác suất nhãn dương trong quận đó.

### Công thức
Với một category $c$:

$$
TE(c) = \bar{y}_c = \frac{1}{n_c}\sum_{i:x_i=c} y_i
$$

Trong thực tế thường dùng **smoothing** để giảm nhiễu khi $n_c$ nhỏ:

$$
TE_{smooth}(c) = \frac{n_c\cdot \bar{y}_c + \alpha\cdot \mu}{n_c + \alpha}
$$

Trong đó:
- $n_c$: số mẫu thuộc category $c$.
- $\bar{y}_c$: trung bình target của category $c$.
- $\mu$: trung bình target toàn bộ tập train.
- $\alpha$: hệ số làm mượt (regularization).

### Cross-validation để tránh target leakage
Nếu tính trung bình target trên toàn bộ tập train rồi gán lại chính train, mô hình sẽ "nhìn thấy" thông tin target của chính dòng đó (leakage).

Cách chuẩn:
1. Chia train thành $K$ fold.
2. Với từng fold validation $k$, chỉ fit mapping target encoding trên $K-1$ fold còn lại.
3. Áp dụng mapping đó cho fold $k$.
4. Khi dự đoán test, fit mapping trên toàn bộ train rồi transform test.

### Ưu điểm so với mã hóa cơ bản
- Khắc phục nhược điểm One-hot về **bùng nổ số chiều**: mỗi cột phân loại chỉ còn 1 cột số.
- Tận dụng trực tiếp thông tin liên quan đến target nên thường mạnh hơn Frequency Encoding và One-hot trong bài toán supervised.
- Hiệu quả với biến nominal có cardinality cao.

### Nhược điểm
- Rất dễ target leakage nếu làm sai quy trình CV.
- Dễ overfit với category hiếm nếu không smoothing/regularization.
- Phụ thuộc vào target, nên không dùng được cho bài toán unsupervised.
- Nhạy với distribution shift (thống kê target thay đổi theo thời gian).

---

## 2.2 Binary Encoding

### Định nghĩa
Binary Encoding kết hợp ý tưởng Label Encoding + chuyển nhị phân:
1. Gán mỗi category một mã số nguyên.
2. Chuyển số nguyên sang biểu diễn nhị phân.
3. Tách từng bit thành các cột 0/1.

### Công thức / Cách thực hiện
Giả sử có $K$ giá trị khác nhau trong cột:
- Cần số bit xấp xỉ:

$$
m = \lceil \log_2(K) \rceil
$$

- Mỗi category sau khi map sang số nguyên sẽ được biểu diễn bằng vector nhị phân độ dài $m$.

Ví dụ: nếu $K=32$ thì chỉ cần khoảng $m=5$ cột bit, thay vì 32 cột như one-hot.

### Cách phân loại dữ liệu phù hợp
- Phù hợp nhất với biến **nominal high-cardinality** (thường > 20 mức).
- Không nên xem Binary Encoding là giữ được quan hệ thứ tự thật giữa các category.

### Ưu điểm so với mã hóa cơ bản
- Khắc phục nhược điểm One-hot về số chiều lớn: số cột tăng theo $\log_2(K)$ thay vì $K$.
- Giảm độ thưa dữ liệu và tiết kiệm bộ nhớ.
- Thường phù hợp hơn One-hot khi cardinality cao.

### Nhược điểm
- Ít trực quan, khó giải thích hơn One-hot.
- Khoảng cách giữa các mã bit không mang ý nghĩa nghiệp vụ rõ ràng.
- Với một số mô hình tuyến tính, chất lượng có thể không ổn định bằng One-hot khi cardinality thấp.

---

## 2.3 Frequency Encoding

### Định nghĩa
Frequency Encoding thay mỗi category bằng **tần suất xuất hiện** (hoặc số lần xuất hiện) của category đó trong tập train.

### Công thức / Cách thực hiện
- Dạng tỷ lệ:

$$
FE(c) = \frac{n_c}{N} = P(X=c)
$$

- Dạng đếm:

$$
FE_{count}(c) = n_c
$$

Trong đó:
- $n_c$: số mẫu thuộc category $c$.
- $N$: tổng số mẫu.

### Cách phân loại dữ liệu phù hợp
- Dùng tốt cho biến nominal có cardinality trung bình đến cao.
- Hữu ích khi cần cách mã hóa nhanh, nhẹ, không dùng target.

### Ưu điểm so với mã hóa cơ bản
- Khắc phục nhược điểm One-hot về bùng nổ chiều: mỗi cột phân loại vẫn chỉ là 1 cột số.
- Không cần target như Target Encoding nên dễ triển khai hơn và ít rủi ro leakage hơn.
- Tính toán nhanh, phù hợp pipeline lớn.

### Nhược điểm
- Các category có cùng tần suất sẽ nhận cùng giá trị, làm mất khả năng phân biệt.
- Không tận dụng thông tin target nên có thể kém hiệu quả hơn Target Encoding trong supervised learning.
- Nhạy với lệch phân phối train-test (data drift), đặc biệt dữ liệu theo thời gian.

---

## 2.4 So sánh nhanh 3 kỹ thuật nâng cao

| Phương pháp | Dùng target? | Số chiều đầu ra | Mạnh khi nào | Rủi ro chính |
|---|---|---|---|---|
| Target Encoding | Có | 1 cột / biến | Supervised + high-cardinality | Leakage, overfit |
| Binary Encoding | Không | $\approx \log_2(K)$ cột / biến | Nominal có rất nhiều mức | Khó diễn giải |
| Frequency Encoding | Không | 1 cột / biến | Cần nhanh, nhẹ, baseline tốt | Mất phân biệt khi cùng tần suất |

### Quy tắc chọn nhanh
1. Nếu có target rõ ràng và cần hiệu năng cao: ưu tiên Target Encoding (kèm CV + smoothing).
2. Nếu không muốn dùng target và cardinality rất lớn: cân nhắc Binary Encoding.
3. Nếu cần phương án đơn giản, nhẹ, nhanh để baseline: chọn Frequency Encoding.